# Qwen uses local Transformers inference in this notebook.
# Keep HF_HUB_OFFLINE unset unless the complete Qwen snapshot is already cached.
# If Hub downloads fail with an expired token, run in a terminal:
#   export HF_HOME=/mnt/cache/taghavi
#   hf auth login --force


In [ ]:
import os
from pathlib import Path

DEFAULT_JAVA_HOME = Path('/home/iataghav/.local/java/jdk-21.0.11+10')
if not os.environ.get('JAVA_HOME') and DEFAULT_JAVA_HOME.is_dir():
    os.environ['JAVA_HOME'] = str(DEFAULT_JAVA_HOME)

if os.environ.get('JAVA_HOME'):
    java_bin = Path(os.environ['JAVA_HOME']) / 'bin'
    if java_bin.is_dir():
        os.environ['PATH'] = f"{java_bin}:" + os.environ.get('PATH', '')

print(f"JAVA_HOME={os.environ.get('JAVA_HOME', '<not set>')}")


In [ ]:
import os
import sys
from pathlib import Path

def set_default_env(name, value):
    if not os.environ.get(name):
        os.environ[name] = str(value)

def find_project_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / 'src' / 'hyde').is_dir() and (path / 'setup.py').is_file():
            return path
    return start

ROOT_DIR = find_project_root()
os.chdir(ROOT_DIR)

src_path = str(ROOT_DIR / 'src')
set_default_env('PYTHONPATH', src_path)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

set_default_env('CUDA_VISIBLE_DEVICES', '4,5,6,7')
set_default_env('GLOBAL_VISIBLE_DEVICES', os.environ['CUDA_VISIBLE_DEVICES'])
set_default_env('TOKENIZERS_PARALLELISM', 'false')

DEFAULT_NOVELHOPQA_BOOKS_ROOT = ROOT_DIR / 'novelhopqa' / 'book-corpus-root'
FALLBACK_NOVELHOPQA_BOOKS_ROOT = Path('/home/iataghav/data/passing_meta_tag/novelhopqa/book-corpus-root')
current_books_root = os.environ.get('NOVELHOPQA_BOOKS_ROOT')
if not current_books_root or not (Path(current_books_root) / 'bookmeta.json').is_file():
    if (DEFAULT_NOVELHOPQA_BOOKS_ROOT / 'bookmeta.json').is_file():
        os.environ['NOVELHOPQA_BOOKS_ROOT'] = str(DEFAULT_NOVELHOPQA_BOOKS_ROOT)
    else:
        os.environ['NOVELHOPQA_BOOKS_ROOT'] = str(FALLBACK_NOVELHOPQA_BOOKS_ROOT)

HF_CACHE_ROOT = os.environ.get('SAADI_HF_CACHE_ROOT') or '/mnt/cache/taghavi'
set_default_env('HF_HOME', HF_CACHE_ROOT)
set_default_env('HF_HUB_CACHE', Path(os.environ['HF_HOME']) / 'hub')
set_default_env('HF_DATASETS_CACHE', Path(os.environ['HF_HOME']) / 'datasets')
set_default_env('TRANSFORMERS_CACHE', Path(os.environ['HF_HOME']) / 'transformers')

if os.environ.get('SAADI_HF_OFFLINE', '').lower() in {'1', 'true', 'yes', 'on'}:
    set_default_env('HF_HUB_OFFLINE', '1')
    set_default_env('TRANSFORMERS_OFFLINE', '1')
else:
    os.environ.pop('HF_HUB_OFFLINE', None)
    os.environ.pop('TRANSFORMERS_OFFLINE', None)

print(f'ROOT_DIR={ROOT_DIR}')
print(f'CUDA_VISIBLE_DEVICES={os.environ["CUDA_VISIBLE_DEVICES"]}')
print(f'HF_HOME={os.environ["HF_HOME"]}')

from pyserini.search.faiss import FaissSearcher
from pyserini.search.lucene import LuceneSearcher
from pyserini.encode import AutoQueryEncoder
from pyserini.search import get_topics, get_qrels
from tqdm import tqdm

### Initialize Contriever Index and Query Encoder
We use [Pyserini](https://github.com/castorini/pyserini) as the search interface for the experiment. Please following the guidance in Pyserini to create Contriever index using the checkpoint from original Contriever work.

In [ ]:
DEFAULT_CONTRIEVER_PATH = Path('/mnt/cache/taghavi/hub/models--facebook--contriever/snapshots/2bd46a25019aeea091fd42d1f0fd4801675cf699')
contriever_path = Path(os.environ.get('CONTRIEVER_PATH', DEFAULT_CONTRIEVER_PATH))
encoder_dir = str(contriever_path) if contriever_path.exists() else 'facebook/contriever'

query_encoder = AutoQueryEncoder(encoder_dir=encoder_dir, pooling='mean')
searcher = FaissSearcher('contriever_msmarco_index/', query_encoder)
corpus = LuceneSearcher.from_prebuilt_index('msmarco-v1-passage')
print(f'Contriever encoder_dir={encoder_dir}')


### Load query and judegments for dl19-passage dataset

In [ ]:
topics = get_topics('dl19-passage')
qrels = get_qrels('dl19-passage')

## Run Contriever

In [ ]:
with open('dl19-contriever-top1000-trec', 'w')  as f:
    for qid in tqdm(topics):
        if qid in qrels:
            query = topics[qid]['title']
            hits = searcher.search(query, k=1000)
            rank = 0
            for hit in hits:
                rank += 1
                f.write(f'{qid} Q0 {hit.docid} {rank} {hit.score} rank\n')

In [ ]:
!python -m pyserini.eval.trec_eval -c -l 2 -m map dl19-passage dl19-contriever-top1000-trec
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.10 dl19-passage dl19-contriever-top1000-trec
!python -m pyserini.eval.trec_eval -c -l 2 -m recall.1000 dl19-passage dl19-contriever-top1000-trec

In [ ]:
import os
from hyde import TransformersGenerator

MODEL_NAME = 'Qwen/Qwen3-30B-A3B-Instruct-2507'
generator = TransformersGenerator(
    MODEL_NAME,
    api_key=os.getenv('HF_TOKEN'),
    n=8,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.8,
    device_map='auto',
    torch_dtype='auto',
    cache_dir=os.environ.get('HF_HUB_CACHE'),
    trust_remote_code=True,
    local_files_only=None,
)
qwen_snapshot = generator.prepare_snapshot()
print(f'Qwen snapshot={qwen_snapshot}')

def call_qwen_api(prompt: str):
    return generator.generate(prompt)

## Run HyDE

In [ ]:
from time import sleep
import numpy as np
import json
from tqdm import tqdm
with open('hyde-dl19-contriever-qwen3-top1000-8rep-trec', 'w') as f, open('hyde-dl19-qwen3-gen.jsonl', 'w') as fgen:
    for qid in tqdm(topics):
        if qid in qrels:
            query = topics[qid]['title']
            print(query)
            prompt = f"""Please write a passage to answer the question
Question: {query}
Passage:"""
            for attempt in range(3):
                try:
                    contexts = [c.strip() for c in call_qwen_api(prompt)] + [query]
                    all_emb_c = []
                    for c in contexts:
                        c_emb = query_encoder.encode(c)
                        all_emb_c.append(np.array(c_emb))
                    all_emb_c = np.array(all_emb_c)
                    avg_emb_c = np.mean(all_emb_c, axis=0)
                    avg_emb_c = avg_emb_c.reshape((1, len(avg_emb_c)))
                    fgen.write(json.dumps({'query_id': qid, 'query': query, 'contexts': contexts})+'\n')
                    break
                except Exception as e:
                    if attempt == 2:
                        raise RuntimeError(f'Qwen generation failed for query id {qid}. Check HF_TOKEN, HF_HUB_CACHE, and whether the full model snapshot is cached/downloadable.') from e
                    sleep(1)
            print(contexts)
            hits = searcher.search(avg_emb_c, k=1000)
            rank = 0
            for hit in hits:
                rank += 1
                f.write(f'{qid} Q0 {hit.docid} {rank} {hit.score} rank\n')

In [ ]:
!python -m pyserini.eval.trec_eval -c -l 2 -m map dl19-passage hyde-dl19-contriever-qwen3-top1000-8rep-trec
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.10 dl19-passage hyde-dl19-contriever-qwen3-top1000-8rep-trec
!python -m pyserini.eval.trec_eval -c -l 2 -m recall.1000 dl19-passage hyde-dl19-contriever-qwen3-top1000-8rep-trec
